# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import os, getpass, duckdb
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
DATA_PATH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'

df = con.sql(f"""
WITH b AS (SELECT MAX(report_date) AS end_d FROM read_parquet('{DATA_PATH}'))
SELECT f.client_hash_id, f.content_hash_id,
  SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
  SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
  AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_prev30,
  STDDEV(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)    AS pos_std_prev30
FROM read_parquet('{DATA_PATH}') f, b
WHERE f.report_date > b.end_d - INTERVAL 60 DAY
GROUP BY 1, 2
HAVING imp_prev30 >= 100
""").df()

df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)
df = df.dropna(subset=['pos_prev30', 'pos_std_prev30']).reset_index(drop=True)

# Cut-offs come from half of the clients; the tiers are scored on the other half.
dev_i, hold_i = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=0)
                     .split(df, groups=df['client_hash_id']))
dev, hold = df.iloc[dev_i], df.iloc[hold_i].copy()
vol_cut = dev['pos_std_prev30'].quantile(0.25)

deep = (hold['pos_prev30'] > 20) & (hold['pos_prev30'] <= 50)   # reason code 1: deep position
unstable = hold['pos_std_prev30'] > vol_cut                      # reason code 2: unstable position
hold['points'] = deep.astype(int) + unstable.astype(int)

names = {2: '1 review first (both codes)', 1: '2 review (one code)', 0: '3 monitor (no code)'}
out = (hold.assign(tier=hold['points'].map(names)).groupby('tier')['is_declining']
       .agg(pages='size', decline_rate='mean').round(3))
print('held-out clients:', hold['client_hash_id'].nunique(),
      '  overall decline rate:', round(hold['is_declining'].mean(), 3))
print(out.to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Ranked actions and reason codes

**The queue (tested on 25 clients held out when setting cut-offs; 53,964 pages; overall decline
rate 68.2%).** Pages are grouped by two reason codes, both measured in the earlier 30-day window,
which does not overlap the window that defines decline:

1. **Deep position:** average position between 21 and 50.
2. **Unstable position:** position volatility above the 25th percentile (cut-off 2.9, learned on
   the other half of the clients).

| Action tier | Reason codes | Pages | Share that declined |
|---|---|---|---|
| 1. Review first | both codes | 14,696 | 76.3% |
| 2. Review | one code | 26,058 | 67.1% |
| 3. Monitor | no code | 13,210 | 61.4% |

**In plain words.** Pages that sit deep in the results and whose position keeps moving declined
most often (76.3%, about 8 points above the overall rate). Pages with neither code declined least
(61.4%). Start the review with tier 1, and put tier 3 on a watch list.

**Caveats.** The position band (21 to 50) was chosen after looking at decline rates by position
band in an earlier notebook on the same data, so this table is partly in-sample. Even the lowest
tier declined more than half of the time, so "monitor" does not mean "safe." These are observed,
directional associations for decision support, not forecasts and not evidence of cause.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

**Who uses it.** A content editor deciding which pages to open first when search visibility is
falling. The ranking is a review queue, not a decision.

**Where it is valid.** Pages with at least 100 impressions in the earlier 30-day window, in
clients like those in the FlyRank release (50 clients, 111,240 pages in scope). Both reason codes
use position measured before the window that defines decline, so they do not overlap the label.
The volatility cut-off was 2.9 (standard deviation of position), learned on the development
clients.

**Where it stops.** It is observed and directional. It does not forecast, it does not show that
any pattern causes a decline, and it says nothing about how Google ranks. Results vary with which
clients are held out, and the cut-off was learned on one half of the clients only.

In [ ]:
print('pages in scope:', len(df), '  clients:', df['client_hash_id'].nunique())
print('smallest earlier-window impressions in scope:', int(df['imp_prev30'].min()))
print('volatility cut-off (learned on development clients):', round(vol_cut, 2))

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review and the no-go list

**A person checks before acting:**
1. The page's recent history, in case the drop is seasonal.
2. The query behind the page's visibility.
3. Whether the page still matches what searchers want.
4. Whether the page has enough impressions for a click-through figure to mean anything. The code
   below shows how many pages in each tier are small (under 1,000 impressions).

**No-go list (never automate):**
- Rewriting, merging, pruning or deleting a page on a score alone.
- Telling a client that a refresh will recover traffic.
- Treating zero-click pages with a few hundred impressions as failures.
- Using hashed IDs as features.
- Publishing page-level rows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring and retrain triggers

**Signs the recommendations have gone stale (proposed triggers):**
- **The tier gap shrinks.** Today the "review first" tier declines about 15 points more often than
  "monitor" (76.3% against 61.4%). If a new 30-day window shows a gap under about 7 points, half
  of today's, re-examine the rule.
- **Held-out-client ROC-AUC falls toward 0.5.**
- **The data changes.** A new release or new clients change the mix of position bands.
- **Cut-offs shift** noticeably when recalculated.

**Rhythm.** Recompute the cut-offs and re-check the tier gap each time a new 30-day window is
complete, and review monthly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the paper

Only the three-row tier summary is exported, to `work/outputs/tier_summary.csv`. I deliberately do
not export a page-level queue: the data-use rules forbid row-level dumps, and pseudonymous IDs
are for grouping only. The paper uses aggregate numbers from the notebooks and does not read this
file.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.